# 03 — Compare Current Run to Baseline

**Run this after every change** to see exactly what it affected — the "did I
break anything?" check. Compares `output/recommendations_local.csv` (your
latest run) against `baseline/recommendations_baseline.csv` (the snapshot you
froze with `scripts/save_baseline.py`).

**Workflow:**
```
python run_pipeline.py             # baseline run, looks right
python scripts/save_baseline.py    # freeze it
... make a change to a model or the runner ...
python run_pipeline.py             # produces a new output/
# open this notebook -> see exactly what changed
```

If nothing has changed, every section below should show 0 differences.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None); pd.set_option('display.width', 160)

CURRENT  = "../output/recommendations_local.csv"
BASELINE = "../baseline/recommendations_baseline.csv"

cur = pd.read_csv(CURRENT, low_memory=False)
base = pd.read_csv(BASELINE, low_memory=False)
cur.columns = [c.strip() for c in cur.columns]
base.columns = [c.strip() for c in base.columns]

import json
manifest = json.load(open("../baseline/manifest.json"))
print(f"Baseline saved: {manifest['saved_at']}")
print(f"CURRENT:  {cur.shape[0]:,} rows")
print(f"BASELINE: {base.shape[0]:,} rows")
print(f"Same columns: {sorted(cur.columns) == sorted(base.columns)}")

## 1. Row-level diff

Match rows on the natural key (clinic + category + species) and see what
changed per row: added, removed, or a value changed.

In [ ]:
KEY = ['CLINIC ID', 'DISEASE CATEGORY', 'SPECIES']

cur_k  = cur.set_index(KEY)
base_k = base.set_index(KEY)

added   = cur_k.index.difference(base_k.index)
removed = base_k.index.difference(cur_k.index)
common  = cur_k.index.intersection(base_k.index)

print(f"Rows added (new clinic/category/species combos):   {len(added):,}")
print(f"Rows removed (existed in baseline, gone now):        {len(removed):,}")
print(f"Rows present in both (comparable):                   {len(common):,}")

## 2. What changed, column by column (among rows present in both)

In [ ]:
COMPARE_COLS = [
    'ACTION RECOMMENDATION', 'HAS SAMPLE', 'DISEASE_CATEGORY_EDUCATION_FLAG',
    'DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION',
    'MONTHLY OPPORTUNITY', 'MONTHLY OPPORTUNITY VOLUME',
    'OPPORTUNITY RANKING WITHIN CLINIC', 'CLINIC TIER', 'CLUSTER',
]
COMPARE_COLS = [c for c in COMPARE_COLS if c in cur.columns and c in base.columns]

c_common = cur_k.loc[common, COMPARE_COLS]
b_common = base_k.loc[common, COMPARE_COLS]

print(f"{'column':45s} {'changed rows':>14s} {'% of common':>12s}")
for col in COMPARE_COLS:
    # numeric cols: flag if the value moved at all; strings: exact mismatch
    if pd.api.types.is_numeric_dtype(c_common[col]) or pd.api.types.is_numeric_dtype(b_common[col]):
        cv = pd.to_numeric(c_common[col], errors='coerce')
        bv = pd.to_numeric(b_common[col], errors='coerce')
        diff = (cv.round(2) != bv.round(2)) & ~(cv.isna() & bv.isna())
    else:
        diff = c_common[col].astype(str) != b_common[col].astype(str)
    n = diff.sum()
    print(f"{col:45s} {n:>14,} {n/len(common)*100:>11.2f}%")

## 3. Distribution shift — opportunity & the top-seller targeting metric

Not just "did values change" but "did they change in a way that matters".

In [ ]:
def opp_stats(df, label):
    o = pd.to_numeric(df['MONTHLY OPPORTUNITY'], errors='coerce')
    print(f"{label:10s} mean {o.mean():8.2f}  median {o.median():6.2f}  max {o.max():10.2f}  zero% {(o==0).mean()*100:5.1f}%")

opp_stats(base, "BASELINE")
opp_stats(cur,  "CURRENT")

In [ ]:
def top_seller_rate(df):
    rank1 = df[df['OPPORTUNITY RANKING WITHIN CLINIC']==1][['CLINIC ID','DISEASE CATEGORY','SPECIES']]
    top_share = (df.sort_values('DISEASE_CATEGORY_SHARE_PCT', ascending=False)
                   .groupby('CLINIC ID').first()[['DISEASE CATEGORY','SPECIES']]
                   .rename(columns={'DISEASE CATEGORY':'top_share_dc','SPECIES':'top_share_sp'}))
    chk = rank1.merge(top_share, on='CLINIC ID')
    chk['same'] = (chk['DISEASE CATEGORY']==chk['top_share_dc']) & (chk['SPECIES']==chk['top_share_sp'])
    return chk['same'].mean()*100

print(f"BASELINE top-seller-is-#1-rec rate: {top_seller_rate(base):.1f}%")
print(f"CURRENT  top-seller-is-#1-rec rate: {top_seller_rate(cur):.1f}%")
print("(Lower is generally better — more gap-finding, less stating the obvious.")
print(" A big jump either way is worth investigating.)")

## 4. The caps — should almost NEVER change unless you edited them on purpose

In [ ]:
def cap_check(df, label):
    sampled = df[df['HAS SAMPLE']==True].groupby('CLINIC SALES REP OR TM')['CLINIC ID'].nunique()
    edu = df[df['DISEASE_CATEGORY_EDUCATION_FLAG']==1]
    print(f"{label:10s} max samples/TM: {sampled.max() if len(sampled) else 0:>3}  "
          f"(cap=14)   |   education recs: {len(edu):>5,}")

cap_check(base, "BASELINE")
cap_check(cur,  "CURRENT")

## 5. Example rows that changed

A handful of actual before/after rows, so you can eyeball whether the change
makes sense — not just that *a* number moved.

In [ ]:
if len(common) > 0 and COMPARE_COLS:
    any_diff = pd.Series(False, index=common)
    for col in COMPARE_COLS:
        if pd.api.types.is_numeric_dtype(c_common[col]) or pd.api.types.is_numeric_dtype(b_common[col]):
            cv = pd.to_numeric(c_common[col], errors='coerce'); bv = pd.to_numeric(b_common[col], errors='coerce')
            any_diff |= (cv.round(2) != bv.round(2)) & ~(cv.isna() & bv.isna())
        else:
            any_diff |= c_common[col].astype(str) != b_common[col].astype(str)

    changed_keys = common[any_diff][:5]
    for k in changed_keys:
        print("Clinic/category/species:", k)
        print("  BASELINE:", b_common.loc[k, COMPARE_COLS].to_dict())
        print("  CURRENT: ", c_common.loc[k, COMPARE_COLS].to_dict())
        print()
else:
    print("No comparable rows or columns to show.")

## Verdict

- **Section 4 (the caps) should show identical numbers** unless you deliberately
  changed a threshold — if they moved unexpectedly, something broke.
- **Sections 2–3** are where your *intended* change should show up — e.g. if you
  enriched clustering, expect `CLINIC TIER`/`CLUSTER` and the opportunity
  distribution to shift; if you only meant to touch clustering, an unexpected
  change in `ACTION RECOMMENDATION` mix is worth a second look.
- Use Section 5 to sanity-check a few real examples, not just the aggregate stats.